# 📦 Optimisation Multi-Étages de la Valeur de Stock — Modèle MILP

---

## Contexte et Problématique

Dans un contexte industriel concurrentiel, la gestion des stocks représente un enjeu financier majeur : **trop de stock immobilise du capital** et génère des risques d'obsolescence, tandis que **trop peu de stock expose à des ruptures** et dégrade le taux de service client.

Ce projet s'inscrit dans une démarche concrète d'amélioration de la performance supply chain d'un site industriel multi-étages (Matières Premières → Semi-Finis → Produits Finis). L'analyse des indicateurs de performance (KPIs) révèle un **surstockage structurel** :

- 💰 **Capital immobilisé excessif** : la valeur totale du stock dépasse significativement la cible fiscale de fin d'année.
- 📉 **Rotation insuffisante** : certaines références accumulent des niveaux bien supérieurs à leur stock de sécurité optimal.
- ⚠️ **Risque d'obsolescence** : un stock trop élevé sur des références à faible rotation peut entraîner des provisions financières.

---

## Objectif du Modèle

L'objectif est de construire un **outil d'aide à la décision** basé sur la Programmation Linéaire Mixte en Nombres Entiers (**MILP — Mixed Integer Linear Programming**) capable de répondre à trois questions opérationnelles :

| Question | Levier |
|----------|--------|
| **Quand commander ?** (MP) | Variable binaire $x_{i,t}$ |
| **Combien commander / produire ?** | Variables continues $O_{i,t}^{MP}$, $P_{j,t}^{SF}$, $P_{l,t}^{PF}$ |
| **La cible fiscale est-elle atteignable ?** | Contrainte $C5$ + statut du solveur |

---

## Architecture du Modèle

Le modèle est structuré en **deux couches** :

1. **Couche théorique — EOQ** : calcule les paramètres économiques de référence (quantité économique de commande, stock de sécurité) qui calibrent le modèle MILP.
2. **Couche décisionnelle — MILP multi-périodes, multi-étages** : optimise la trajectoire de stock de juin à septembre, sous toutes les contraintes opérationnelles réelles (lead times, temps de cycle, BOM, MOQ, MLS, fermeture fournisseur en août).

```
MP ──(achat, lead time LT_i)──► Stock MP
Stock MP ──(production, BOM a_ij)──► Stock SF    (temps de cycle ct_j)
Stock SF ──(production, BOM b_jl)──► Stock PF    (temps de cycle ct_l)
Stock PF ──(vente)──────────────────► Demande client D_{l,t}
```

---

## Principe de séparation Modèle / Données

> Ce notebook suit le principe **AMPL / GAMS** : le modèle mathématique est défini de façon **purement abstraite** (sections 1 à 4), sans aucune valeur numérique.
> Toutes les données concrètes sont regroupées **en section 5 uniquement**.
> Pour adapter le modèle à des données réelles (SAP), il suffit de modifier la section 5.


---
## 0. Imports

In [171]:
# Pyomo : librairie de modélisation mathématique (AbstractModel, Var, Param, Constraint, Objective...)
from pyomo.environ import *
from IPython.display import display

# Pandas : manipulation de tableaux de données (affichage des résultats)
import pandas as pd

# NumPy : calculs numériques (formules EOQ, racines carrées)
import numpy as np


---
## 1. LES PARAMÈTRES

### Notations mathématiques

#### Indices (ensembles abstraits)

| Indice | Ensemble | Signification |
|--------|----------|---------------|
| $i$ | $MP$ | Références Matières Premières |
| $j$ | $SF$ | Références Semi-Finies |
| $l$ | $PF$ | Références Produits Finis |
| $t$ | $\{1, \ldots, T\}$ | Périodes mensuelles, $T$ = fin d'horizon fiscal (fin septembre) |

#### Paramètres économiques globaux

| Symbole | Description |
|---------|-------------|
| $Target$ | Cible de valeur de stock totale en fin d'horizon (\$) |
| $\tau$ | Taux de possession annuel (%) |
| $z_{\alpha}$ | Coefficient du niveau de service cible $\alpha$ (ex : $z_{0.95} = 1.65$) |
| $M$ | Grand nombre pour la linéarisation *big-M* (contraintes MOQ / MLS) |
| $V_{max}$ | Capacité de stockage globale (m³) |

#### Paramètres par référence

| Symbole | Étage | Description |
|---------|-------|-------------|
| $p_i^{MP},\, p_j^{SF},\, p_l^{PF}$ | MP / SF / PF | Valeur unitaire (\$/unité) |
| $S_{i,0}^{MP},\, S_{j,0}^{SF},\, S_{l,0}^{PF}$ | tous | Stock initial réel à $t=0$ (extrait SAP) |
| $LT_i$ | MP | Lead time fournisseur (mois) |
| $ct_j,\, ct_l$ | SF / PF | Temps de cycle de production (mois) |
| $MOQ_i$ | MP | Quantité minimale de commande |
| $MLS_j,\, MLS_l$ | SF / PF | Taille de lot minimale de production |
| $c_j,\, c_l$ | SF / PF | Charge unitaire de production (heures/unité) |
| $v_i^{MP},\, v_j^{SF},\, v_l^{PF}$ | tous | Volume unitaire de stockage (m³/unité) |
| $SS_k$ | tous | Stock de sécurité (calculé via EOQ, injecté en data) |

#### Paramètres de flux

| Symbole | Description |
|---------|-------------|
| $D_{l,t}$ | Demande client du PF $l$ en période $t$ (seul étage avec demande externe) |
| $OC_{i,t}^{MP}$ | Commandes fournisseur déjà passées, arrivant en $t$ (figées, extraites SAP) |
| $WIP_{j,t}^{SF},\, WIP_{l,t}^{PF}$ | Encours de production déjà lancés, sortant en $t$ (figés, extraits SAP) |
| $a_{ij}$ | BOM MP $\to$ SF : qté de MP $i$ nécessaire pour 1 unité de SF $j$ |
| $b_{jl}$ | BOM SF $\to$ PF : qté de SF $j$ nécessaire pour 1 unité de PF $l$ |
| $Cap_t^{SF},\, Cap_t^{PF}$ | Capacité de production disponible en $t$ (heures) |

> ⚠️ Aucune valeur numérique dans cette section — uniquement la déclaration des paramètres abstraits Pyomo.


In [172]:
# Initialisation du modèle abstrait Pyomo
# AbstractModel : le modèle est défini sans données — elles seront injectées via DataPortal (section 5)
model = AbstractModel()

# ── Horizon temporel ─────────────────────────────────────────────────────────
model.T        = Param(within=PositiveIntegers)   # T : dernière période (fin d'horizon fiscal)
model.PER      = RangeSet(1, model.T)             # {1, 2, ..., T} : ensemble des périodes
model.t_juillet = Param(within=PositiveIntegers)  # indice de juillet (dernier mois avant fermeture)
model.AoutSet  = Set(within=model.PER)            # sous-ensemble : période(s) de fermeture fournisseur

# ── Ensembles de références par étage ────────────────────────────────────────
model.MP = Set()   # ensemble des indices Matières Premières
model.SF = Set()   # ensemble des indices Semi-Finis
model.PF = Set()   # ensemble des indices Produits Finis

# ── Paramètres économiques globaux ───────────────────────────────────────────
model.TARGET  = Param()   # cible de valeur de stock fin d'horizon ($)
model.tau     = Param()   # taux de possession annuel (ex : 0.20 = 20 %)
model.z_alpha = Param()   # coefficient niveau de service (ex : 1.65 pour alpha=95%)
model.M       = Param()   # big-M pour linéarisation des contraintes MOQ / MLS
model.V_max   = Param()   # capacité de stockage globale (m³)

# ── Paramètres par référence — Matières Premières ────────────────────────────
model.prix_MP       = Param(model.MP)   # p_i^MP : valeur unitaire ($)
model.stock_init_MP = Param(model.MP)   # S_i,0^MP : stock initial réel (t=0)
model.LT            = Param(model.MP)   # LT_i : lead time fournisseur (mois)
model.MOQ           = Param(model.MP)   # MOQ_i : quantité minimale de commande
model.volume_MP     = Param(model.MP)   # v_i^MP : volume unitaire (m³/unité)
model.SS_MP         = Param(model.MP)   # SS_i : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Semi-Finis ────────────────────────────────────
model.prix_SF       = Param(model.SF)   # p_j^SF : valeur unitaire ($)
model.stock_init_SF = Param(model.SF)   # S_j,0^SF : stock initial réel (t=0)
model.ct_SF         = Param(model.SF)   # ct_j : temps de cycle production (mois)
model.MLS_SF        = Param(model.SF)   # MLS_j : taille de lot minimale
model.charge_SF     = Param(model.SF)   # c_j : charge unitaire (heures/unité)
model.volume_SF     = Param(model.SF)   # v_j^SF : volume unitaire (m³/unité)
model.SS_SF         = Param(model.SF)   # SS_j : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Produits Finis ────────────────────────────────
model.prix_PF       = Param(model.PF)   # p_l^PF : valeur unitaire ($)
model.stock_init_PF = Param(model.PF)   # S_l,0^PF : stock initial réel (t=0)
model.ct_PF         = Param(model.PF)   # ct_l : temps de cycle production (mois)
model.MLS_PF        = Param(model.PF)   # MLS_l : taille de lot minimale
model.charge_PF     = Param(model.PF)   # c_l : charge unitaire (heures/unité)
model.volume_PF     = Param(model.PF)   # v_l^PF : volume unitaire (m³/unité)
model.SS_PF         = Param(model.PF)   # SS_l : stock de sécurité (pré-calculé EOQ)

# ── Demande client (uniquement sur les Produits Finis) ───────────────────────
# D_{l,t} : seul l'étage PF a une demande externe réelle (commandes clients)
model.demande = Param(model.PF, model.PER)

# ── Pipeline déjà engagé (données figées, extraites de SAP) ──────────────────
# Ces paramètres représentent des flux DÉJÀ DÉCIDÉS avant le démarrage du modèle.
# Le modèle ne peut pas les modifier — ce sont des constantes.
model.OC     = Param(model.MP, model.PER)   # OC_{i,t} : commandes MP en transit
model.WIP_SF = Param(model.SF, model.PER)   # WIP_{j,t}^SF : encours production SF
model.WIP_PF = Param(model.PF, model.PER)   # WIP_{l,t}^PF : encours production PF

# ── Nomenclature BOM (Bill of Materials) ─────────────────────────────────────
# default=0 : si un couple (i,j) ou (j,l) n'est pas déclaré en data, la valeur est 0
model.a = Param(model.MP, model.SF, default=0)   # a_{ij} : MP -> SF
model.b = Param(model.SF, model.PF, default=0)   # b_{jl} : SF -> PF

# ── Capacités de production par période ──────────────────────────────────────
# Cap_t^SF / Cap_t^PF : exprimées en heures disponibles par mois
# Réduite en août (congés fournisseurs et production)
model.Cap_SF = Param(model.PER)   # capacité atelier SF (heures/mois)
model.Cap_PF = Param(model.PER)   # capacité atelier PF (heures/mois)


---
## 2. LES VARIABLES DE DÉCISION

### Formulation mathématique

Le modèle distingue deux conventions temporelles importantes :

- Les **stocks** $S_{k,t}$ sont des **photos de fin de période** $t$ (après tous les mouvements du mois).
- Les **décisions** ($O$, $P$, $x$, $y$) sont **lancées en début de période** $t$ et produisent leur effet après un délai ($LT_i$ ou $ct_k$).

| Variable | Domaine | Convention | Signification |
|----------|---------|------------|---------------|
| $S_{i,t}^{MP}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock MP $i$ en fin de période $t$ |
| $S_{j,t}^{SF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock SF $j$ en fin de période $t$ |
| $S_{l,t}^{PF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock PF $l$ en fin de période $t$ |
| $O_{i,t}^{MP}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **commandée** (achat MP $i$) lancée en début de $t$ |
| $P_{j,t}^{SF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** SF $j$ en début de $t$ |
| $P_{l,t}^{PF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** PF $l$ en début de $t$ |
| $x_{i,t}$ | $\{0, 1\}$ | Début de $t$ | **1** si une commande est passée pour MP $i$ en $t$, **0** sinon |
| $y_{j,t}^{SF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour SF $j$ en $t$, **0** sinon |
| $y_{l,t}^{PF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour PF $l$ en $t$, **0** sinon |

**Note sur le décalage temporel :**
Ce qui est commandé/lancé en début de $t$ n'arrive en stock qu'en fin de $t + LT_i$ (ou $t + ct_k$).
C'est ce décalage qui est encodé dans les contraintes de bilan **C1 / C2 / C3**.


In [173]:
# ── Stocks en fin de période t ────────────────────────────────────────────────
# domain=NonNegativeReals impose S >= 0 (contrainte C13 implicite)
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # stock MP fin de t
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # stock SF fin de t
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # stock PF fin de t

# ── Quantités commandées (MP) et lancées en production (SF, PF) ───────────────
# Lancées en début de t — l'effet en stock est décalé par LT_i ou ct_k (voir C1/C2/C3)
model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # O_{i,t}^MP
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # P_{j,t}^SF
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # P_{l,t}^PF

# ── Variables binaires de déclenchement ──────────────────────────────────────
# domain=Binary impose x, y in {0, 1} (contrainte C13 implicite)
model.x    = Var(model.MP, model.PER, domain=Binary)   # x_{i,t} : commande MP déclenchée ?
model.y_SF = Var(model.SF, model.PER, domain=Binary)   # y_{j,t} : OF SF déclenché ?
model.y_PF = Var(model.PF, model.PER, domain=Binary)   # y_{l,t} : OF PF déclenché ?


---
## 3. LA FONCTION OBJECTIF

### Formulation mathématique

$$\min\; Z = \sum_{t=1}^{T}\left(
  \sum_{i \in MP} p_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,t}^{PF}
\right)$$

**Interprétation :**
On minimise la **valeur financière totale du stock immobilisé**, cumulée sur tout l'horizon (juin → septembre).

- Multiplier chaque quantité en stock par son prix unitaire donne la **valeur immobilisée** à cette période.
- Sommer sur toutes les périodes (et pas seulement la dernière) évite un déstockage artificiel concentré en fin d'horizon : le modèle est incité à réduire le stock **dès le début**, pas seulement en septembre.
- La hiérarchie des prix $p^{PF} > p^{SF} > p^{MP}$ fait que le modèle **priorise naturellement la réduction des stocks de produits finis**, qui immobilisent le plus de valeur ajoutée.


In [174]:
# ── Fonction objectif ─────────────────────────────────────────────────────────
# Minimiser la somme des valeurs de stock (prix * quantité) sur tous les étages et toutes les périodes
def cost_rule(m):
    return (
        # Valeur immobilisée en Matières Premières
        sum(m.prix_MP[i] * m.S_MP[i, t] for i in m.MP for t in m.PER)
        # Valeur immobilisée en Semi-Finis
      + sum(m.prix_SF[j] * m.S_SF[j, t] for j in m.SF for t in m.PER)
        # Valeur immobilisée en Produits Finis
      + sum(m.prix_PF[l] * m.S_PF[l, t] for l in m.PF for t in m.PER)
    )

# sense=minimize : on minimise (par opposition à maximize)
model.cost = Objective(rule=cost_rule, sense=minimize)


---
## 3 (suite). LES CONTRAINTES C1 → C13

---

### C1 — Bilan de stock Matière Première

$$S_{i,t}^{MP} = \underbrace{S_{i,t-1}^{MP}}_{\text{stock précédent}}
+ \underbrace{OC_{i,t}^{MP}}_{\substack{\text{pipeline SAP}\\\text{(figé)}}}
+ \underbrace{O_{i,\;t-LT_i}^{MP}}_{\substack{\text{décision modèle}\\\text{(décalée de }LT_i\text{)}}}
- \underbrace{\sum_{j \in SF} a_{ij}\, P_{j,t}^{SF}}_{\text{consommation par la production SF}}
\quad \forall\, i \in MP,\; t \in T$$

**Points clés :**
- Si $t = 1$ : $S_{i,t-1}^{MP} = S_{i,0}^{MP}$ (stock initial réel SAP) → **C4 intégrée ici**.
- Si $t - LT_i < 1$ : la commande décidée par le modèle n'est pas encore arrivée → terme nul.
- La consommation de MP a lieu **au moment du lancement** de la production SF (pas à sa sortie).

---

### C2 — Bilan de stock Semi-Fini

$$S_{j,t}^{SF} = S_{j,t-1}^{SF}
+ \underbrace{WIP_{j,t}^{SF}}_{\substack{\text{encours déjà}\\\text{lancé (SAP)}}}
+ \underbrace{P_{j,\;t-ct_j}^{SF}}_{\substack{\text{décision modèle}\\\text{(décalée de }ct_j\text{)}}}
- \underbrace{\sum_{l \in PF} b_{jl}\, P_{l,t}^{PF}}_{\text{consommation par la production PF}}
\quad \forall\, j \in SF,\; t \in T$$

---

### C3 — Bilan de stock Produit Fini

$$S_{l,t}^{PF} = S_{l,t-1}^{PF}
+ \underbrace{WIP_{l,t}^{PF}}_{\text{encours (SAP)}}
+ \underbrace{P_{l,\;t-ct_l}^{PF}}_{\text{décision modèle}}
- \underbrace{D_{l,t}}_{\substack{\text{demande client}\\\text{(seul étage avec}\\\text{demande externe)}}}
\quad \forall\, l \in PF,\; t \in T$$

---

### C4 — Initialisation des stocks

$$S_{i,0}^{MP} = \text{stock\_init\_MP}[i], \quad
S_{j,0}^{SF} = \text{stock\_init\_SF}[j], \quad
S_{l,0}^{PF} = \text{stock\_init\_PF}[l]$$

> **Intégrée dans C1/C2/C3** via la condition $t = 1$ — pas de contrainte Pyomo séparée.

---

### C5 — Cible de valeur de stock fin d'année fiscale

$$\sum_{i \in MP} p_i^{MP}\, S_{i,T}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,T}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,T}^{PF}
\leq Target$$

**Contrainte principale du modèle** : la valeur totale du stock en fin d'horizon (fin septembre) doit être inférieure ou égale à la cible fiscale $Target$.

---

### C6 — Taux de service minimum (Produit Fini uniquement)

$$S_{l,t}^{PF} \geq SS_l \quad \forall\, l \in PF,\; t \in T$$

Le stock de PF ne doit jamais descendre en dessous du stock de sécurité $SS_l$ (calculé via EOQ), garantissant un niveau de service $\alpha$ vis-à-vis des clients.

---

### C7 — Capacité de production

$$\underbrace{\sum_{j \in SF} c_j\, P_{j,t}^{SF}}_{\text{charge totale générée}} \leq Cap_t^{SF} \quad \forall\, t$$

$$\underbrace{\sum_{l \in PF} c_l\, P_{l,t}^{PF}}_{\text{charge totale générée}} \leq Cap_t^{PF} \quad \forall\, t$$

- $c_k$ = **charge unitaire** (heures nécessaires pour produire 1 unité) — paramètre par référence.
- $Cap_t$ = **capacité disponible** (heures totales du mois) — réduite en août pour les congés.

---

### C8 — Capacité de stockage globale (volume physique)

$$\sum_{i \in MP} v_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} v_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} v_l^{PF}\, S_{l,t}^{PF}
\leq V_{max} \quad \forall\, t$$

Le volume total occupé ne doit pas dépasser la capacité physique de l'entrepôt.

---

### C9 — MOQ et liaison binaire — Achat Matière Première

$$O_{i,t}^{MP} \geq MOQ_i \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(quantité min si commande)}$$
$$O_{i,t}^{MP} \leq M \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(force zéro si pas de commande)}$$

- Si $x_{i,t} = 0$ → $O_{i,t}^{MP} = 0$ (aucune commande)
- Si $x_{i,t} = 1$ → $O_{i,t}^{MP} \geq MOQ_i$ (commande ≥ quantité minimale)

---

### C10 — MLS et liaison binaire — Production SF et PF

$$P_{j,t}^{SF} \geq MLS_j \cdot y_{j,t}^{SF}, \quad P_{j,t}^{SF} \leq M \cdot y_{j,t}^{SF} \quad \forall\, j, t$$
$$P_{l,t}^{PF} \geq MLS_l \cdot y_{l,t}^{PF}, \quad P_{l,t}^{PF} \leq M \cdot y_{l,t}^{PF} \quad \forall\, l, t$$

Même logique que C9 : si un OF est lancé ($y = 1$), la quantité produite doit respecter la taille de lot minimale $MLS$.

---

### C11 — Fermeture fournisseur en août

$$O_{i,t}^{MP} = 0 \quad \forall\, i \in MP,\; t \in t_{ao\hat{u}t}$$

Les fournisseurs sont fermés en août : aucune nouvelle commande MP ne peut être passée.

---

### C12 — Couverture de stock avant fermeture

$$S_{i,\, t_{juillet}}^{MP} \geq \sum_{t \in t_{ao\hat{u}t}} \sum_{j \in SF} a_{ij}\, P_{j,t}^{SF} \quad \forall\, i \in MP$$

Le stock MP disponible fin juillet doit couvrir **toute la consommation prévue en août**, puisqu'aucun réapprovisionnement ne sera possible.
Cette contrainte force le modèle à constituer un **stock tampon en juillet** → pic visible sur la trajectoire.

---

### C13 — Non-négativité et domaine des variables

$$S^{MP},\, S^{SF},\, S^{PF},\, O^{MP},\, P^{SF},\, P^{PF} \geq 0 \qquad x_{i,t},\, y_{j,t}^{SF},\, y_{l,t}^{PF} \in \{0, 1\}$$

> **Intégrée dans la déclaration des variables** via `domain=NonNegativeReals` et `domain=Binary` — pas de contrainte Pyomo séparée.


In [175]:
# ── C1 — Bilan de stock Matière Première ─────────────────────────────────────
# S[i,t] = S[i,t-1] + OC[i,t] + O[i, t-LT_i] - sum_j(a[i,j] * P_SF[j,t])
def c1_rule(m, i, t):
    # Stock de la période précédente (ou stock initial si t=1 → intègre C4)
    S_prev = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]

    # Réception issue d'une commande DÉCIDÉE par le modèle, lancée t-LT_i périodes avant
    t_cmd     = t - m.LT[i]
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0  # nul si hors horizon

    # Consommation de MP i par toutes les productions SF lancées en t (via BOM)
    conso = sum(m.a[i, j] * m.P_SF[j, t] for j in m.SF if m.a[i, j] > 0)

    return m.S_MP[i, t] == S_prev + m.OC[i, t] + reception - conso

model.C1_Bilan_MP = Constraint(model.MP, model.PER, rule=c1_rule)

# ── C2 — Bilan de stock Semi-Fini ─────────────────────────────────────────────
# S[j,t] = S[j,t-1] + WIP_SF[j,t] + P_SF[j, t-ct_j] - sum_l(b[j,l] * P_PF[l,t])
def c2_rule(m, j, t):
    S_prev = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]

    # Sortie de production SF décidée par le modèle, lancée t-ct_j périodes avant
    t_lanc    = t - m.ct_SF[j]
    reception = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0

    # Consommation de SF j par toutes les productions PF lancées en t (via BOM)
    conso = sum(m.b[j, l] * m.P_PF[l, t] for l in m.PF if m.b[j, l] > 0)

    return m.S_SF[j, t] == S_prev + m.WIP_SF[j, t] + reception - conso

model.C2_Bilan_SF = Constraint(model.SF, model.PER, rule=c2_rule)

# ── C3 — Bilan de stock Produit Fini ──────────────────────────────────────────
# S[l,t] = S[l,t-1] + WIP_PF[l,t] + P_PF[l, t-ct_l] - D[l,t]
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]

    # Sortie de production PF décidée par le modèle, lancée t-ct_l périodes avant
    t_lanc    = t - m.ct_PF[l]
    reception = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0

    # Demande client externe : seul flux de sortie non contrôlable par le modèle
    return m.S_PF[l, t] == S_prev + m.WIP_PF[l, t] + reception - m.demande[l, t]

model.C3_Bilan_PF = Constraint(model.PF, model.PER, rule=c3_rule)

# C4 — Initialisation : intégrée dans C1/C2/C3 via la condition t==1 (pas de contrainte séparée)

# ── C5 — Cible de valeur de stock fin d'année fiscale ─────────────────────────
# sum_i(p_i * S_MP[i,T]) + sum_j(p_j * S_SF[j,T]) + sum_l(p_l * S_PF[l,T]) <= TARGET
def c5_rule(m):
    return (
        sum(m.prix_MP[i] * m.S_MP[i, m.T] for i in m.MP)
      + sum(m.prix_SF[j] * m.S_SF[j, m.T] for j in m.SF)
      + sum(m.prix_PF[l] * m.S_PF[l, m.T] for l in m.PF)
      <= m.TARGET
    )
model.C5_Cible_Fiscale = Constraint(rule=c5_rule)

# ── C6 — Taux de service minimum (Produits Finis uniquement) ──────────────────
# S_PF[l,t] >= SS_l pour tout l, t
def c6_rule(m, l, t):
    return m.S_PF[l, t] >= m.SS_PF[l]   # SS_PF[l] calculé via EOQ, injecté en data
model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)

# ── C7 — Capacité de production (charge totale <= capacité disponible) ─────────
# sum_j(c_j * P_SF[j,t]) <= Cap_SF[t]   pour tout t
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j] * m.P_SF[j, t] for j in m.SF) <= m.Cap_SF[t]
model.C7_Capacite_SF = Constraint(model.PER, rule=c7_sf_rule)

# sum_l(c_l * P_PF[l,t]) <= Cap_PF[t]   pour tout t
def c7_pf_rule(m, t):
    return sum(m.charge_PF[l] * m.P_PF[l, t] for l in m.PF) <= m.Cap_PF[t]
model.C7_Capacite_PF = Constraint(model.PER, rule=c7_pf_rule)

# ── C8 — Capacité de stockage globale (volume physique) ───────────────────────
# sum(v_i*S_MP + v_j*S_SF + v_l*S_PF) <= V_max   pour tout t
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i] * m.S_MP[i, t] for i in m.MP)
      + sum(m.volume_SF[j] * m.S_SF[j, t] for j in m.SF)
      + sum(m.volume_PF[l] * m.S_PF[l, t] for l in m.PF)
      <= m.V_max
    )
model.C8_Capacite_Stockage = Constraint(model.PER, rule=c8_rule)

# ── C9 — MOQ et liaison binaire — Achat MP ────────────────────────────────────
# O_MP[i,t] >= MOQ_i * x[i,t]   : si commande, quantité >= MOQ
def c9a_rule(m, i, t):
    return m.O_MP[i, t] >= m.MOQ[i] * m.x[i, t]
model.C9a_MOQ_min = Constraint(model.MP, model.PER, rule=c9a_rule)

# O_MP[i,t] <= M * x[i,t]   : si pas de commande (x=0), force O_MP=0
def c9b_rule(m, i, t):
    return m.O_MP[i, t] <= m.M * m.x[i, t]
model.C9b_MOQ_max = Constraint(model.MP, model.PER, rule=c9b_rule)

# ── C10 — MLS et liaison binaire — Production SF / PF ─────────────────────────
# Même logique que C9 mais pour les lancements de production

# P_SF[j,t] >= MLS_j * y_SF[j,t]
def c10a_sf_rule(m, j, t):
    return m.P_SF[j, t] >= m.MLS_SF[j] * m.y_SF[j, t]
model.C10a_MLS_SF_min = Constraint(model.SF, model.PER, rule=c10a_sf_rule)

# P_SF[j,t] <= M * y_SF[j,t]
def c10b_sf_rule(m, j, t):
    return m.P_SF[j, t] <= m.M * m.y_SF[j, t]
model.C10b_MLS_SF_max = Constraint(model.SF, model.PER, rule=c10b_sf_rule)

# P_PF[l,t] >= MLS_l * y_PF[l,t]
def c10a_pf_rule(m, l, t):
    return m.P_PF[l, t] >= m.MLS_PF[l] * m.y_PF[l, t]
model.C10a_MLS_PF_min = Constraint(model.PF, model.PER, rule=c10a_pf_rule)

# P_PF[l,t] <= M * y_PF[l,t]
def c10b_pf_rule(m, l, t):
    return m.P_PF[l, t] <= m.M * m.y_PF[l, t]
model.C10b_MLS_PF_max = Constraint(model.PF, model.PER, rule=c10b_pf_rule)

# ── C11 — Fermeture fournisseur en août (aucune commande possible) ─────────────
# O_MP[i,t] = 0  pour tout i, t dans AoutSet
def c11_rule(m, i, t):
    return m.O_MP[i, t] == 0
model.C11_Fermeture_Fournisseur = Constraint(model.MP, model.AoutSet, rule=c11_rule)

# ── C12 — Couverture de stock avant fermeture ──────────────────────────────────
# S_MP[i, t_juillet] >= sum_{t in AoutSet} sum_j(a[i,j] * P_SF[j,t])
# Stock fin juillet >= toute la consommation MP prévue en août
def c12_rule(m, i):
    conso_aout = sum(
        m.a[i, j] * m.P_SF[j, t]
        for t in m.AoutSet
        for j in m.SF
        if m.a[i, j] > 0
    )
    return m.S_MP[i, m.t_juillet] >= conso_aout
model.C12_Couverture_Aout = Constraint(model.MP, rule=c12_rule)

# C13 — Non-négativité et domaine binaire :
#        déjà intégrés via domain=NonNegativeReals et domain=Binary dans la déclaration des variables


---
## 4. RÉSOLUTION (fonction générique)

Cette fonction encapsule l'appel au solveur.
Elle sera appelée dans la section 5 (données), après la construction de l'instance concrète.


In [176]:
def resoudre(instance, solveur="cbc", verbose=True):
    """
    Résout l'instance concrète du modèle MILP.

    Paramètres
    ----------
    instance : instance Pyomo concrète (créée via model.create_instance(data))
    solveur  : nom du solveur à utiliser (défaut : 'cbc', open-source)
    verbose  : afficher les logs du solveur (défaut : True)

    Retourne
    --------
    resultats : objet Pyomo contenant le statut et les informations de résolution
    """
    opt       = SolverFactory(solveur)   # initialise le solveur
    resultats = opt.solve(instance, tee=verbose)
    return resultats


---
## 5. GÉNÉRATION LOGIQUE DES DONNÉES MANQUANTES

Fonctions définies ici, appelées dans la section 6 une fois les listes MP/SF/PF connues.

In [177]:
import math, os
import pandas as pd
import numpy as np
from pyomo.environ import DataPortal, value as pyo_value

# ─────────────────────────────────────────────────────────────────────────────
# NORMALISATION DES PN SAP
# Règle unique appliquée partout : strip + majuscules.
# Les zéros de gauche sont CONSERVÉS tels quels (format SAP natif).
# ─────────────────────────────────────────────────────────────────────────────
def norm(pn):
    """Normalisation standard d'un PN SAP : strip + majuscules. Zéros conservés."""
    return str(pn).strip().upper()

# ─────────────────────────────────────────────────────────────────────────────
def generer_volumes(prix_MP, prix_SF, prix_PF):
    """Volumes unitaires de stockage (m³/unité) par règle industrielle."""
    ref_MP = max(prix_MP.values()) if prix_MP else 50
    ref_SF = max(prix_SF.values()) if prix_SF else 200
    ref_PF = max(prix_PF.values()) if prix_PF else 800
    return (
        {i: round(0.003 * (p / ref_MP) ** 0.4, 4) for i, p in prix_MP.items()},
        {j: round(0.025 * (p / ref_SF) ** 0.5, 4) for j, p in prix_SF.items()},
        {l: round(0.070 * (p / ref_PF) ** 0.5, 4) for l, p in prix_PF.items()},
    )

def generer_charges_et_cycles(prix_SF, prix_PF):
    """Charges (h/unité) et temps de cycle (mois) par règle industrielle."""
    charge_SF = {j: round(1.0 + p / 300, 2) for j, p in prix_SF.items()}
    charge_PF = {l: round(2.0 + p / 400, 2) for l, p in prix_PF.items()}
    ct_SF     = {j: 2 if charge_SF[j] >= 3.0 else 1 for j in prix_SF}
    ct_PF     = {l: 1 for l in prix_PF}
    return charge_SF, charge_PF, ct_SF, ct_PF

def generer_capacites(T, t_aout, oee=0.85, taux_aout=0.30):
    """Capacités de production (h/mois). Base : 2 lignes SF / 1 ligne PF x 22j x 8h x OEE.
    En août (fermeture/congés), la capacité est réduite à taux_aout (30% par défaut)
    au lieu du calcul plein — bug corrigé : l'ancienne version multipliait par
    99999999 au lieu d'appliquer un OEE de 0.85, et augmentait (au lieu de réduire)
    la capacité en août."""
    base_SF = round(2 * 8 * 22 * oee)
    base_PF = round(1 * 8 * 22 * oee)
    Cap_SF = {t: round(base_SF * taux_aout) if t in t_aout else base_SF for t in T}
    Cap_PF = {t: round(base_PF * taux_aout) if t in t_aout else base_PF for t in T}
    return Cap_SF, Cap_PF

def generer_couts_fixes(prix_MP, prix_SF, prix_PF):
    """Coûts fixes de passation/lancement (pour calibration EOQ)."""
    return (
        {i: max(100, round(p * 2.5)) for i, p in prix_MP.items()},
        {j: max(150, round(p * 1.5)) for j, p in prix_SF.items()},
        {l: max(200, round(p * 1.0)) for l, p in prix_PF.items()},
    )

def generer_sigma_D(stock_init_MP, stock_init_SF, stock_init_PF, demande, L, J, I, T):
    """Écart-types de la demande par référence."""
    sigma = {}
    for l in L:
        dem_moy = sum(demande.get((l, t), 0) for t in T) / len(T)
        sigma[l] = max(5, round(dem_moy * 0.15))
    for j in J:
        sigma[j] = max(5, round(stock_init_SF.get(j, 0) * 0.12))
    for i in I:
        sigma[i] = max(5, round(stock_init_MP.get(i, 0) * 0.08))
    return sigma

print("✅  Section 5 — Fonctions de génération définies")


✅  Section 5 — Fonctions de génération définies


---
## 6. MISE EN PLACE DE LA DATA

---
### 6.0 — Dossier et horizon temporel

In [178]:
DOSSIER_DATA  = r"/content/sample_data"   # ◄ adapter si nécessaire
def f(nom): return os.path.join(DOSSIER_DATA, nom)

T             = [1, 2, 3, 4]
T_max         = 4
t_juillet     = 2
t_aout        = [3]
noms_periodes = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}

print(f"✅  Dossier : {os.path.abspath(DOSSIER_DATA)}")
print(f"   Périodes : {[noms_periodes[t] for t in T]}")


✅  Dossier : /content/sample_data
   Périodes : ['Juin', 'Juillet', 'Août', 'Septembre']


---
### 6.1 — Référentiel produits (`Stock.xlsx`)

Types SAP : `ZROH` = MP | `ZHLB` = SF | `ZFRT` = PF

In [179]:
df_stock = pd.read_excel(f("/content/sample_data/PN ,STOCK, PRIX EN STOCK,TYPE PN.XLSX"), sheet_name="Sheet1")
df_stock.columns = df_stock.columns.str.strip()

# Normalisation des PN (règle unique : strip + majuscules)
df_stock["Material"] = df_stock["Material"].apply(norm)

# Suppression ligne TOTAL et doublons
df_stock = df_stock[df_stock["Material"].notna() & (df_stock["Material"] != "TOTAL")]
df_stock = df_stock.drop_duplicates(subset="Material", keep="first").set_index("Material")

# Séparation par type
I = df_stock[df_stock["Material Type"] == "ZROH"].index.tolist()
J = df_stock[df_stock["Material Type"] == "ZHLB"].index.tolist()
L = df_stock[df_stock["Material Type"] == "ZFRT"].index.tolist()

# Dictionnaires de référence (clés déjà normalisées)
prix_MP       = df_stock.loc[I, "Prix_Unitaire"].astype(float).to_dict()
stock_init_MP = df_stock.loc[I, "Unrestricted Stock"].astype(float).to_dict()
prix_SF       = df_stock.loc[J, "Prix_Unitaire"].astype(float).to_dict()
stock_init_SF = df_stock.loc[J, "Unrestricted Stock"].astype(float).to_dict()
prix_PF       = df_stock.loc[L, "Prix_Unitaire"].astype(float).to_dict()
stock_init_PF = df_stock.loc[L, "Unrestricted Stock"].astype(float).to_dict()
type_par_pn   = df_stock["Material Type"].to_dict()   # utilisé par la BOM

# Valeur initiale du stock
val_MP  = sum(prix_MP[i]  * stock_init_MP[i]  for i in I)
val_SF  = sum(prix_SF[j]  * stock_init_SF[j]  for j in J)
val_PF  = sum(prix_PF[l]  * stock_init_PF[l]  for l in L)
val_tot = val_MP + val_SF + val_PF

print("=" * 62)
print("  STOCK INITIAL")
print("=" * 62)
print(f"  MP  : {len(I):4d} références  →  {val_MP:>14,.0f}  $")
print(f"  SF  : {len(J):4d} références  →  {val_SF:>14,.0f}  $")
print(f"  PF  : {len(L):4d} références  →  {val_PF:>14,.0f}  $")
print(f"  {'─'*46}")
print(f"  TOTAL                      →  {val_tot:>14,.0f}  $")
print("=" * 62)


  STOCK INITIAL
  MP  :  281 références  →         558,929  $
  SF  : 3714 références  →       3,115,515  $
  PF  : 1052 références  →       1,138,717  $
  ──────────────────────────────────────────────
  TOTAL                      →       4,813,161  $


---
### 6.2 — Paramètres globaux (`Parametres_Modele.xlsx`)

In [180]:
df_p = pd.read_excel(f("/content/sample_data/Parametres_Modele.xlsx"), sheet_name="Parametres_Globaux")
df_p.columns = df_p.columns.str.strip()
params = df_p.set_index("Paramètre")["Valeur"].to_dict()

TARGET  = float(params["TARGET"])
tau     = float(params["tau"])
z_alpha = float(params["z_alpha"])
M       = float(params["M_bigM"])
V_max   = float(params["V_max"])

print("=" * 62)
print("  PARAMÈTRES GLOBAUX")
print("=" * 62)
print(f"  TARGET  (cible fiscale)  : {TARGET:>14,.0f}  $")
print(f"  V_max   (capacité stock) : {V_max:>14,.0f}  m³")
print(f"  tau     (possession)     : {tau*100:>13.0f}  %/an")
print(f"  z_alpha (niveau service) : {z_alpha:>14.2f}  → α ≈ 95%")
print(f"  Écart à réduire          : {val_tot - TARGET:>14,.0f}  $")
print("=" * 62)


  PARAMÈTRES GLOBAUX
  TARGET  (cible fiscale)  :    999,999,999  $
  V_max   (capacité stock) :    999,999,999  m³
  tau     (possession)     :            20  %/an
  z_alpha (niveau service) :           1.65  → α ≈ 95%
  Écart à réduire          :   -995,186,838  $


---
### 6.3 — Lead Times (`LEAD TIME FOURNISSEUR AVEC PN.xlsx`)

Conversion jours → mois : $LT_{mois} = \max(1,\, \lceil LT_{jours}/30 \rceil)$.

In [181]:
df_lt = pd.read_excel(f("/content/sample_data/LEAD TIME FOURNISSEUR AVEC PN.xlsx"))
df_lt.columns = df_lt.columns.str.strip()
df_lt["PN"] = df_lt["PN"].apply(norm)

# Si un PN a plusieurs fournisseurs → max du lead time (scénario prudent)
lt_jours = df_lt.groupby("PN")["LEAD TIME"].max().to_dict()

LT = {}
for i in I:
    jours = lt_jours.get(i, 30)   # 30 jours par défaut si manquant
    LT[i] = max(1, math.ceil(float(jours) / 30))

mp_manquants = [i for i in I if i not in lt_jours]
if mp_manquants:
    print(f"⚠️  {len(mp_manquants)} MP sans LT → 1 mois par défaut")

print(f"✅  Lead Times : {len(LT)} MP chargés")
df_lt_aff = pd.DataFrame([
    {"PN": i, "LT (jours)": lt_jours.get(i,"—"), "LT (mois)": LT[i]}
    for i in I
]).set_index("PN")
display(df_lt_aff)


⚠️  15 MP sans LT → 1 mois par défaut
✅  Lead Times : 281 MP chargés


,LT (jours),LT (mois)
PN,,
065073-000,21,1
1-1573168-0,56,2
1-1573168-1,56,2
1-1573168-2,56,2
1-1573168-3,56,2
...,...,...
9-1806155-7,128,5
9-1806155-9,83,3
9-705422-9,248,9


---
### 6.4 — MOQ et MLS (`PN MLS.xls`)

In [182]:
df_moq = pd.read_excel(f("/content/sample_data/PN MLS.xls"))
df_moq.columns = df_moq.columns.str.strip()
df_moq["Material"] = df_moq["Material"].apply(norm)
moq_series = df_moq.set_index("Material")["Cur. Minimum lot size"]

def get_moq(pn, defaut=1.0):
    v = moq_series.get(pn, None)
    if v is None or (isinstance(v, float) and math.isnan(v)): return defaut
    return float(v)

MOQ    = {i: get_moq(i) for i in I}
MLS_SF = {j: get_moq(j) for j in J}
MLS_PF = {l: get_moq(l) for l in L}

print(f"✅  MOQ  : {len(MOQ)} MP   |  MLS SF : {len(MLS_SF)}   |  MLS PF : {len(MLS_PF)}")


✅  MOQ  : 281 MP   |  MLS SF : 3714   |  MLS PF : 1052


---
### 6.5 — Demande client (`DEMAND ANALYSSIS.xlsx`)

Colonne PN : `MATERIAL` — Colonnes semaines : format `2026/27` (année/semaine ISO).
Agrégation hebdo → mensuelle par somme.

In [183]:
def semaine_vers_t(col):
    """'2026/27' → t du modèle (1=Juin, 2=Juil, 3=Août, 4=Sept). None si hors horizon."""
    try:
        annee, sem = str(col).strip().split("/")
        sem = int(sem)
    except Exception:
        return None
    # Semaine ISO → mois civil approx (chaque mois ≈ 4.33 semaines)
    mois = math.ceil(sem / 4.33)
    mois = max(1, min(12, mois))
    return {6: 1, 7: 2, 8: 3, 9: 4}.get(mois, None)

df_dem = pd.read_excel(f("/content/sample_data/DEMAND ANALYSSIS.xlsx"))
df_dem.columns = [str(c).strip() for c in df_dem.columns]
df_dem["MATERIAL"] = df_dem["MATERIAL"].apply(norm)
df_dem = df_dem[df_dem["MATERIAL"].isin(L)].set_index("MATERIAL")

# Colonnes semaines dans l'horizon
week_to_t_local = {col: semaine_vers_t(col) for col in df_dem.columns
                   if semaine_vers_t(col) is not None}

if not week_to_t_local:
    raise ValueError(
        "Aucune colonne semaine dans l'horizon juin-septembre.\n"
        "Format attendu : '2026/23' … '2026/39'\n"
        "Colonnes dispo : " + str(list(df_dem.columns[:10]))
    )

# Conversion numérique et agrégation
for col in week_to_t_local:
    df_dem[col] = pd.to_numeric(df_dem[col], errors="coerce").fillna(0)

demande = {}
for l in L:
    for t in T:
        cols_t = [w for w, v in week_to_t_local.items() if v == t]
        if l in df_dem.index and cols_t:
            demande[(l, t)] = int(df_dem.loc[l, cols_t].sum())
        else:
            demande[(l, t)] = 0

df_agg = pd.DataFrame({noms_periodes[t]: {l: demande[(l,t)] for l in L} for t in T})
df_agg.index.name = "PN (PF)"
print(f"✅  Demande agrégée ({len(week_to_t_local)} semaines → 4 mois) :")
display(df_agg)


✅  Demande agrégée (20 semaines → 4 mois) :


,Juin,Juillet,Août,Septembre
PN (PF),,,,
002172-000,0,0,0,0
007250-000,0,0,0,0
017076-000,0,0,0,0
022016-000,0,100,460,100
026942-000,0,0,0,0
...,...,...,...,...
F33905-000,0,0,0,0
F36265-000,0,0,0,0
F45196-000,0,0,0,0


---
### 6.6 — Pipeline SAP : WIP et achats en cours (`WIP_Achats_En_Cours.xlsx`)

In [184]:
OC     = {i: {t: 0.0 for t in T} for i in I}
WIP_SF = {j: {t: 0.0 for t in T} for j in J}
WIP_PF = {l: {t: 0.0 for t in T} for l in L}

xl = f("/content/sample_data/WIP_Achats_En_Cours.xlsx")

def charger_pipeline(sheet, col_pn, col_qte, dico, liste_valide):
    try:
        df = pd.read_excel(xl, sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        df[col_pn] = df[col_pn].apply(norm)
        for _, row in df.iterrows():
            pn = row[col_pn]
            t  = int(row.get("Mois Modèle (t)", 0))
            q  = float(row.get(col_qte, 0) or 0)
            if pn in dico and t in T:
                dico[pn][t] += q
    except Exception as e:
        print(f"⚠️  Feuille '{sheet}' : {e}")

charger_pipeline("Achats_En_Cours_MP",  "PN",    "Qté Commandée", OC,     I)
charger_pipeline("WIP_Semi_Finis",      "PN SF", "Qté En-Cours",  WIP_SF, J)
charger_pipeline("WIP_Produits_Finis",  "PN PF", "Qté En-Cours",  WIP_PF, L)

tot_oc  = sum(v for i in OC     for v in OC[i].values())
tot_wsf = sum(v for j in WIP_SF for v in WIP_SF[j].values())
tot_wpf = sum(v for l in WIP_PF for v in WIP_PF[l].values())
print(f"✅  Pipeline SAP chargé")
print(f"   OC total    : {tot_oc:,.0f} unités")
print(f"   WIP SF total: {tot_wsf:,.0f} unités")
print(f"   WIP PF total: {tot_wpf:,.0f} unités")


✅  Pipeline SAP chargé
   OC total    : 259,838 unités
   WIP SF total: 415,819 unités
   WIP PF total: 370,924 unités


---
### 6.7 — Nomenclature BOM (`BOM_SAP.xlsx`)

Structure du fichier SAP :

| Colonne | Signification |
|---------|---------------|
| `Parent` | PN du produit parent (SF ou PF) |
| `Base quantity` | Quantité de référence de la nomenclature |
| `Material / Doc` | PN du composant (MP ou SF) |
| `Quantity` | Quantité du composant pour fabriquer `Base quantity` unités |

$$q_{unitaire} = \frac{\text{Quantity}}{\text{Base quantity}}$$


In [185]:
df_bom = pd.read_excel(f("/content/sample_data/ZBOM.XLSX"))
df_bom.columns = df_bom.columns.str.strip()

df_bom["Parent"]         = df_bom["Parent"].apply(norm)
df_bom["Material / Doc"] = df_bom["Material / Doc"].apply(norm)
df_bom["Base quantity"]  = pd.to_numeric(df_bom["Base quantity"], errors="coerce").fillna(1)
df_bom["Quantity"]       = pd.to_numeric(df_bom["Quantity"],       errors="coerce").fillna(0)
df_bom["qty_unitaire"]   = df_bom["Quantity"] / df_bom["Base quantity"]

# Matrices BOM — initialisées à 0 pour tous les couples connus
a_bom = {i: {j: 0.0 for j in J} for i in I}   # MP → SF
b_bom = {j: {l: 0.0 for l in L} for j in J}   # SF → PF

n_a = n_b = n_ignore = 0
for _, row in df_bom.iterrows():
    parent    = row["Parent"]
    composant = row["Material / Doc"]
    qty       = float(row["qty_unitaire"])

    t_parent    = type_par_pn.get(parent,    "")
    t_composant = type_par_pn.get(composant, "")

    if t_composant == "ZROH" and t_parent == "ZHLB":   # MP → SF
        if composant in a_bom and parent in a_bom[composant]:
            a_bom[composant][parent] += qty;  n_a += 1
        else: n_ignore += 1
    elif t_composant == "ZHLB" and t_parent == "ZFRT":  # SF → PF
        if composant in b_bom and parent in b_bom[composant]:
            b_bom[composant][parent] += qty;  n_b += 1
        else: n_ignore += 1
    else:
        n_ignore += 1

print(f"✅  BOM chargée : {len(df_bom)} lignes")
print(f"   Liens MP→SF : {n_a}  |  Liens SF→PF : {n_b}  |  Ignorées : {n_ignore}")

if J and I:
    display(pd.DataFrame(a_bom).T.fillna(0).loc[
        [i for i in I if any(a_bom[i][j]>0 for j in J)],
        [j for j in J if any(a_bom[i][j]>0 for i in I)]
    ])
if J and L:
    display(pd.DataFrame(b_bom).T.fillna(0).loc[
        [j for j in J if any(b_bom[j][l]>0 for l in L)],
        [l for l in L if any(b_bom[j][l]>0 for j in J)]
    ])


✅  BOM chargée : 11452 lignes
   Liens MP→SF : 1517  |  Liens SF→PF : 2413  |  Ignorées : 7522


,1-1018731-1,1-1229227-1,1-1350835-3,1-1350835-6,1-1429272-4,1-1429406-3,1-1429406-6,1-1429406-7,1-1429406-8,1-1429406-9,...,EF1881-000,EF1883-000,EF1884-000,EF1885-000,EF1887-000,EF1888-000,EJ1196-000,EJ1197-000,ES0084-000,F11432-001
1-1573168-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1-1573168-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1-1573168-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1-1573168-3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1-1573168-4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9-1806155-4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9-1806155-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9-705422-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
K1017261,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,1-1510481-6,1-1510619-3,1-2037763-6,1-2512548-1,1-709221-3,1-739146-1,1229804-2,1229804-4,1273966-1,1306035-1,...,A27453-000,CJ8922-000,CN7849-000,CU6151-000,CZ4118-000,E52939-000,EP1670-000,ER2586-000,ER4935-000,F32558-000
1-1018731-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
1-1701210-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
1-1806869-6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
1-1806871-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
1-1806884-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9-739158-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
A02852-000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00604
A53901-000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
D89522-001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000


---
### 6.8 — Application des données générées logiquement

In [186]:
volume_MP, volume_SF, volume_PF           = generer_volumes(prix_MP, prix_SF, prix_PF)
charge_SF, charge_PF, ct_SF, ct_PF       = generer_charges_et_cycles(prix_SF, prix_PF)
Cap_SF, Cap_PF                           = generer_capacites(T, t_aout)
C_passation, C_lancement_SF, C_lancement_PF = generer_couts_fixes(prix_MP, prix_SF, prix_PF)
sigma_D = generer_sigma_D(stock_init_MP, stock_init_SF, stock_init_PF,
                          demande, L, J, I, T)

print("✅  Données générées logiquement :")
rows = []
for i in I: rows.append({"PN":i,"Type":"MP","Vol(m³/u)":volume_MP[i],"Charge(h/u)":"—","Cycle":"—","σ_D":sigma_D[i]})
for j in J: rows.append({"PN":j,"Type":"SF","Vol(m³/u)":volume_SF[j],"Charge(h/u)":charge_SF[j],"Cycle":ct_SF[j],"σ_D":sigma_D[j]})
for l in L: rows.append({"PN":l,"Type":"PF","Vol(m³/u)":volume_PF[l],"Charge(h/u)":charge_PF[l],"Cycle":ct_PF[l],"σ_D":sigma_D[l]})
display(pd.DataFrame(rows).set_index("PN"))

print("\n  Capacités de production :")
df_cap = pd.DataFrame({"Cap SF (h)":Cap_SF,"Cap PF (h)":Cap_PF})
df_cap.index = [noms_periodes[t] for t in T]
display(df_cap)


✅  Données générées logiquement :


,Type,Vol(m³/u),Charge(h/u),Cycle,σ_D
PN,,,,,
065073-000,MP,0.0001,—,—,69
1-1573168-0,MP,0.0009,—,—,5
1-1573168-1,MP,0.0008,—,—,5
1-1573168-2,MP,0.0007,—,—,5
1-1573168-3,MP,0.0007,—,—,5
...,...,...,...,...,...
F33905-000,PF,0.0039,2.01,1,5
F36265-000,PF,0.0019,2.0,1,5
F45196-000,PF,0.0021,2.0,1,5



  Capacités de production :


,Cap SF (h),Cap PF (h)
Juin,35199999648,17599999824
Juillet,35199999648,17599999824
Août,3519999929600000352,1759999964800000176
Septembre,35199999648,17599999824


---\n### 6.9 — Calibration EOQ — Stocks de sécurité $SS_k$\n\n$$Q_k^* = \sqrt{\\frac{2\\,D_k\\,C_k}{h_k}}, \\quad SS_k = z_\\alpha \\cdot \\sigma_{D_k} \\cdot \\sqrt{LT_k \\text{ ou } ct_k}$$

In [187]:
# Demandes annuelles par cascade BOM
D_an_PF = {l: max(1, sum(demande.get((l,t),0) for t in T) * 3) for l in L}
D_an_SF = {j: max(1, sum(b_bom[j][l] * D_an_PF[l] for l in L)) for j in J}
D_an_MP = {i: max(1, sum(a_bom[i][j] * D_an_SF[j] for j in J)) for i in I}

def calc_eoq(D, C, h):
    """Calcule EOQ en sécurisant contre division par zéro et valeurs infinies."""
    h_safe = max(h, 0.01)          # plancher sur h
    val    = 2 * D * C / h_safe
    return math.sqrt(val) if (val > 0 and math.isfinite(val)) else 0.0

SS   = {}
rows = []

for i in I:
    h   = tau * max(prix_MP[i], 0.01)
    Qs  = calc_eoq(D_an_MP[i], C_passation[i], h)
    ss  = z_alpha * sigma_D[i] * math.sqrt(max(LT[i], 1))
    SS[i] = ss
    rows.append({"PN":i,"Étage":"MP","h($/an)":round(h,2),
                 "EOQ*":int(Qs),"SS":int(ss)})   # int() au lieu de round()

for j in J:
    h   = tau * max(prix_SF[j], 0.01)
    Qs  = calc_eoq(D_an_SF[j], C_lancement_SF[j], h)
    ss  = z_alpha * sigma_D[j] * math.sqrt(max(ct_SF[j], 1))
    SS[j] = ss
    rows.append({"PN":j,"Étage":"SF","h($/an)":round(h,2),
                 "EOQ*":int(Qs),"SS":int(ss)})

for l in L:
    h   = tau * max(prix_PF[l], 0.01)
    Qs  = calc_eoq(D_an_PF[l], C_lancement_PF[l], h)
    ss  = z_alpha * sigma_D[l] * math.sqrt(max(ct_PF[l], 1))
    SS[l] = ss
    rows.append({"PN":l,"Étage":"PF","h($/an)":round(h,2),
                 "EOQ*":int(Qs),"SS":int(ss)})

SS_MP = {i: SS[i] for i in I}
SS_SF = {j: SS[j] for j in J}
SS_PF = {l: SS[l] for l in L}

# Rapport PN avec données manquantes
pns_prix_nul = [k for k,p in {**prix_MP,**prix_SF,**prix_PF}.items() if p == 0]
if pns_prix_nul:
    print(f"⚠️  {len(pns_prix_nul)} PN avec prix = 0 → EOQ non significatif : {pns_prix_nul[:5]}")

print("✅  Stocks de sécurité (EOQ) :")
display(pd.DataFrame(rows).set_index("PN"))

✅  Stocks de sécurité (EOQ) :


,Étage,h($/an),EOQ*,SS
PN,,,,
065073-000,MP,0.03,76,113
1-1573168-0,MP,5.96,62,11
1-1573168-1,MP,3.80,7,11
1-1573168-2,MP,3.00,8,11
1-1573168-3,MP,2.71,8,11
...,...,...,...,...
F33905-000,PF,0.98,20,8
F36265-000,PF,0.25,40,8
F45196-000,PF,0.30,36,8


---
### 6.10 — DataPortal Pyomo → Instance concrète

> **Note Pyomo** : les paramètres scalaires doivent être passés sous `{None: valeur}` dans un `DataPortal`.

In [188]:
data = DataPortal()

# ── Scalaires : {None: valeur} obligatoire pour Pyomo DataPortal ──────────────
data['T']         = {None: T_max}
data['t_juillet'] = {None: t_juillet}
data['AoutSet']   = t_aout          # Set → liste directe
data['MP']        = I
data['SF']        = J
data['PF']        = L

data['TARGET']    = {None: TARGET}
data['tau']       = {None: tau}
data['z_alpha']   = {None: z_alpha}
data['M']         = {None: M}
data['V_max']     = {None: V_max}

# ── Dictionnaires indexés → directement ───────────────────────────────────────
data['prix_MP']       = prix_MP
data['stock_init_MP'] = stock_init_MP
data['LT']            = LT
data['MOQ']           = MOQ
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP

data['prix_SF']       = prix_SF
data['stock_init_SF'] = stock_init_SF
data['ct_SF']         = ct_SF
data['MLS_SF']        = MLS_SF
data['charge_SF']     = charge_SF
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

data['prix_PF']       = prix_PF
data['stock_init_PF'] = stock_init_PF
data['ct_PF']         = ct_PF
data['MLS_PF']        = MLS_PF
data['charge_PF']     = charge_PF
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

data['demande'] = {(l, t): demande[(l, t)] for l in L for t in T}
data['OC']      = {(i, t): OC[i][t]        for i in I for t in T}
data['WIP_SF']  = {(j, t): WIP_SF[j][t]    for j in J for t in T}
data['WIP_PF']  = {(l, t): WIP_PF[l][t]    for l in L for t in T}
data['a']       = {(i, j): a_bom[i][j]     for i in I for j in J if a_bom[i][j] > 0}
data['b']       = {(j, l): b_bom[j][l]     for j in J for l in L if b_bom[j][l] > 0}
data['Cap_SF']  = Cap_SF
data['Cap_PF']  = Cap_PF

instance = model.create_instance(data)

print("=" * 62)
print("  ✅  INSTANCE CONCRÈTE — PRÊTE À RÉSOUDRE")
print("=" * 62)
print(f"  MP  : {len(I):4d}  |  SF : {len(J):4d}  |  PF : {len(L):4d}")
print(f"  Valeur initiale : {val_tot:>14,.0f}  $")
print(f"  Cible TARGET    : {TARGET:>14,.0f}  $")
print(f"  Écart à réduire : {val_tot-TARGET:>14,.0f}  $")
from pyomo.core import Constraint
print(f"  Contraintes     : {sum(1 for _ in instance.component_objects(Constraint, active=True))}")
print("=" * 62)


  ✅  INSTANCE CONCRÈTE — PRÊTE À RÉSOUDRE
  MP  :  281  |  SF : 3714  |  PF : 1052
  Valeur initiale :      4,813,161  $
  Cible TARGET    :    999,999,999  $
  Écart à réduire :   -995,186,838  $
  Contraintes     : 16


In [189]:
!apt-get install -y coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


---
### 6.11 — Diagnostic pré-résolution (vérifications de faisabilité évidentes)

Avant même de lancer le solveur, on vérifie les cas où l'infaisabilité est **mathématiquement
certaine** dès la lecture des données — en particulier la contrainte C6 (stock de sécurité PF)
appliquée dès t=1, où aucune production n'a encore eu le temps d'arriver.


In [ ]:
print("Vérification C6 — stock de sécurité PF exigible dès t=1")
print("-" * 70)
alerte_C6 = []
for l in L:
    max_possible = stock_init_PF[l] + WIP_PF[l][1] - demande.get((l, 1), 0)
    if max_possible < SS_PF[l]:
        alerte_C6.append(l)
        print(f"  ❌ {l:<15s} stock max atteignable en t=1 = {max_possible:>10.0f}"
              f"   <   SS requis = {SS_PF[l]:>10.0f}")

if not alerte_C6:
    print("  ✅ Aucun problème détecté sur C6 à t=1")
else:
    print(f"\n⚠️  {len(alerte_C6)} référence(s) PF rendent le modèle infaisable dès t=1 : {alerte_C6}")
    print("   Cause : le stock réel de départ est déjà sous le seuil de sécurité calculé,")
    print("   et aucune production ne peut arriver avant la fin de la période 1 (ct_PF >= 1).")


> **Deux façons de traiter ce cas**, à choisir selon ce qui a du sens pour le projet :
>
> 1. **Ne pas exiger le SS dès t=1** (le stock initial est une donnée subie, pas une décision) —
>    la contrainte C6 ne s'applique qu'à partir de t=2, ce qui laisse au modèle le temps de
>    reconstituer le stock de sécurité.
> 2. **Garder C6 dès t=1** mais vérifier/corriger le stock initial ou la demande de la période 1
>    dans le fichier source.
>
> L'option retenue ici est la n°1 (la plus courante en pratique), implémentée en modifiant la
> règle `c6_rule` (section 3, cellule C6) :
> ```python
> def c6_rule(m, l, t):
>     if t == 1:
>         return Constraint.Skip
>     return m.S_PF[l, t] >= m.SS_PF[l]
> model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)
> ```


---
### 6.12 — Recherche automatique de la cause racine (si le modèle reste infaisable)

Si le diagnostic ci-dessus ne suffit pas à expliquer l'infaisabilité, cette cellule désactive
**une contrainte à la fois** et relance le solveur : la ou les contraintes dont la désactivation
rend le modèle de nouveau faisable sont la cause racine. (Peut prendre quelques minutes selon la
taille du modèle.)


In [ ]:
from pyomo.core import Constraint as _Constraint

def tester_sans(instance, nom_contrainte, solveur="cbc"):
    comp = getattr(instance, nom_contrainte)
    comp.deactivate()
    res = SolverFactory(solveur).solve(instance, tee=False)
    comp.activate()
    return res.solver.termination_condition

noms_contraintes = [c.name for c in instance.component_objects(_Constraint, active=True)]

print("Recherche de la cause racine — désactivation contrainte par contrainte :")
print("-" * 70)
coupables = []
for nom in noms_contraintes:
    statut = tester_sans(instance, nom)
    if statut == TerminationCondition.optimal:
        coupables.append(nom)
        print(f"  🎯 {nom:<28s} : {statut}  <-- redevient FAISABLE si on l'enlève !")
    else:
        print(f"     {nom:<28s} : {statut}")

print("-" * 70)
if coupables:
    print(f"Contrainte(s) responsable(s) de l'infaisabilité : {coupables}")
else:
    print("Aucune contrainte isolée ne suffit à elle seule à expliquer l'infaisabilité")
    print("(la cause est probablement une combinaison de plusieurs contraintes).")


In [190]:
# Configuration du solveur avec affichage des logs (tee=True)
opt = SolverFactory('cbc', executable='/usr/bin/cbc')

print("--- Lancement du calcul avec affichage des détails ---")

# tee=True permet de voir ce que le solveur "pense" en temps réel
resultats = opt.solve(instance, tee=True)

print("\n" + "=" * 55)
print("  ANALYSE DU RÉSULTAT")
print("=" * 55)

condition = resultats.solver.termination_condition

if condition == TerminationCondition.optimal:
    print(f"✅ OPTIMAL : Solution trouvée !")
    print(f"Z* (Coût total) : {value(instance.cost):,.0f} $")
elif condition == TerminationCondition.infeasible:
    print(f"❌ INFAISABLE : Les contraintes sont impossibles à respecter.")
    print("Conseil : Augmente ton TARGET ou vérifie tes capacités.")
else:
    print(f"⚠️ STATUT : {condition}")
    print("Le solveur n'a pas pu conclure. Regarde les messages au-dessus.")
print("=" * 55)

--- Lancement du calcul avec affichage des détails ---
Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -printingOptions all -import /tmp/tmp8s16s6pk.pyomo.lp -stat=1 -solve -solu /tmp/tmp8s16s6pk.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
Presolve determined that the problem was infeasible with tolerance of 1e-08
Presolved model looks infeasible - will use unpresolved
Original problem has 20188 integers (20188 of which binary)
==== 40376 zero objective 3785 different
==== absolute objective values 3785 different
==== for integers 20188 zero objective 1 different
20188 variables have objective of 0
==== for integers absolute objective values 1 different
20188 variables have objective of 0
===== end objective counts


Problem has 65347 rows, 60564 columns (20188 with objective) and 187381 elements
There are 180 singletons with no objective 
Column breakdown:
40376 of type 0.0->inf, 0 of type

  - termination condition: infeasible
  - message from solver: <undefined>



  ANALYSE DU RÉSULTAT
❌ INFAISABLE : Les contraintes sont impossibles à respecter.
Conseil : Augmente ton TARGET ou vérifie tes capacités.


---
## 7. RÉSOLUTION ET RÉSULTATS

In [191]:
resultats = resoudre(instance, solveur="cbc", verbose=False)

print("=" * 55)
print("  STATUT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut  : {resultats.solver.termination_condition}")
print(f"  Z*      : {pyo_value(instance.cost):>14,.0f}  $")
print("=" * 55)


ApplicationError: No executable found for solver 'cbc'

In [ ]:
T_max_v = int(pyo_value(instance.T))

# Vérification de la cible C5
val_fin = (
    sum(instance.prix_MP[i] * pyo_value(instance.S_MP[i, T_max_v]) for i in I)
  + sum(instance.prix_SF[j] * pyo_value(instance.S_SF[j, T_max_v]) for j in J)
  + sum(instance.prix_PF[l] * pyo_value(instance.S_PF[l, T_max_v]) for l in L)
)
print(f"  Valeur stock fin {noms_periodes[T_max_v]} : {val_fin:>14,.0f} $")
print(f"  Cible TARGET               : {TARGET:>14,.0f} $")
print(f"  Cible respectée            : {'✅ Oui' if val_fin <= TARGET else '❌ Non'}")


In [ ]:
import matplotlib.pyplot as plt

valeurs = []
for t in T:
    v_mp = sum(instance.prix_MP[i] * pyo_value(instance.S_MP[i, t]) for i in I)
    v_sf = sum(instance.prix_SF[j] * pyo_value(instance.S_SF[j, t]) for j in J)
    v_pf = sum(instance.prix_PF[l] * pyo_value(instance.S_PF[l, t]) for l in L)
    valeurs.append({"Période": noms_periodes[t], "MP": v_mp, "SF": v_sf, "PF": v_pf,
                    "Total": v_mp + v_sf + v_pf})
df_val = pd.DataFrame(valeurs).set_index("Période")

print("Valeur de stock par période et par étage :")
display(df_val.applymap(lambda x: f"{x:,.0f} $"))

fig, ax = plt.subplots(figsize=(10, 5))
for col, color in [("MP","steelblue"),("SF","darkorange"),("PF","seagreen")]:
    ax.plot(df_val.index, df_val[col], marker="o", label=col, color=color)
ax.plot(df_val.index, df_val["Total"], marker="s", lw=2.5, color="black", label="Total")
ax.axhline(TARGET, color="red", ls="--", lw=1.5, label=f"Cible TARGET ({TARGET:,.0f} $)")
ax.set_title("Trajectoire de la valeur de stock — Juin à Septembre", fontsize=12, fontweight="bold")
ax.set_xlabel("Période"); ax.set_ylabel("Valeur ($)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
